In [2]:
import pandas as pd
import matplotlib.pyplot as plt

## LOADING THE DATA

In [4]:
movies = pd.read_csv('data/TMDB_230000_movies.csv')
movies.head()

,title,genres,overview,cast,director,producers,writers,poster_path,year_norm
0,Ariel,"Comedy, Drama, Romance, Crime",A Finnish man goes to the city to find a job a...,"Tarja Keinänen, Marja Packalén, Kari Helaseppä...",Aki Kaurismäki,Aki Kaurismäki,Aki Kaurismäki,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg,0.681481
1,Shadows in Paradise,"Comedy, Drama, Romance","Nikander, a rubbish collector and would-be ent...","Ari Korhonen, Mari Rantasila, Erkki Rissanen, ...",Aki Kaurismäki,Mika Kaurismäki,Aki Kaurismäki,/nj01hspawPof0mJmlgfjuLyJuRN.jpg,0.666667
2,Four Rooms,Comedy,It's Ted the Bellhop's first night on the job....,"Marisa Tomei, Kimberly Blair, Madonna, Tamlyn ...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Lawrence Bender, Quentin Tarantino, Alexandre ...","Quentin Tarantino, Robert Rodriguez, Alexandre...",/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg,0.733333
3,Judgment Night,"Action, Crime, Thriller","Four young friends, while taking a shortcut en...","Doug Wert, Angela Alvarado, Everlast, Sean O'G...",Stephen Hopkins,"Marilyn Vance, Gene Levy, Lloyd Segan","Lewis Colick, Jere Cunningham",/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg,0.718519
4,Star Wars,"Adventure, Action, Science Fiction",Princess Leia is captured and held hostage by ...,"Jeremy Sinden, Michael Leader, Derek Lyons, Al...",George Lucas,"Gary Kurtz, George Lucas",George Lucas,/6FfCtAuVAW8XJjZ7eWeLibRLWTw.jpg,0.600000


## EXPLORING DATA

In [5]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237898 entries, 0 to 237897
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   title        237898 non-null  object 
 1   genres       237898 non-null  object 
 2   overview     237898 non-null  object 
 3   cast         237898 non-null  object 
 4   director     237898 non-null  object 
 5   producers    237898 non-null  object 
 6   writers      237898 non-null  object 
 7   poster_path  237898 non-null  object 
 8   year_norm    237898 non-null  float64
dtypes: float64(1), object(8)
memory usage: 16.3+ MB


In [6]:
movies.isnull().sum()

title          0
genres         0
overview       0
cast           0
director       0
producers      0
writers        0
poster_path    0
year_norm      0
dtype: int64

## CONTENT-BASED RECOMMENDATION

This one looks at the features of the movies themselves — genre, director, cast, keywords. If you liked Inception, it finds other movies with similar characteristics. Similarity will be computed using Cosine Similarity on a TF-IDF matrix built from the metadata.

In [7]:
movies['features'] = movies['genres'] + ' ' + movies['overview'] + ' ' + movies['cast'] + ' ' + movies['director'] + ' ' + movies['producers'] + ' ' + movies['writers'] 

🎯 TF‑IDF = “How important is this word for describing this movie?”
TF‑IDF stands for:

TF — Term Frequency  
How often a word appears in a document (movie description).

IDF — Inverse Document Frequency  
How rare that word is across the entire dataset.

Then it multiplies them:

TF-IDF
=
TF
×
IDF
🧠 Why this matters for your recommender
TF‑IDF gives high weight to words that are:

frequent in this movie

rare in other movies

And low weight to words that are:

common everywhere (“the”, “movie”, “story”, “man”, “life”)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=50000  # optional, keeps it manageable
)

tfidf_matrix = tfidf.fit_transform(movies['features'])


In [ ]:
from sklearn.metrics.pairwise import linear_kernel

# faster than cosine_similarity
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)


In [ ]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    return movies['title'].iloc[movie_indices]

In [ ]:
get_recommendations("Inception")